# Titanic Dataset

## Setup

In [1]:
import pandas as pd
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import arff

In [2]:
filepath="data/Titanic.arff"
with open(filepath, "r", encoding="utf-8") as f:
    dataset = arff.load(f)
df = pd.DataFrame(dataset["data"],columns=[col[0] for col in dataset["attributes"]])

## Handling missing data

### Wstępna inspekcja

In [62]:
for atr in dataset.get("attributes"):
    print(atr)
print(f"\nLiczba obserwachji: {len(df)}.")

('pclass', 'NUMERIC')
('survived', ['0', '1'])
('name', 'STRING')
('sex', ['female', 'male'])
('age', 'NUMERIC')
('sibsp', 'NUMERIC')
('parch', 'NUMERIC')
('ticket', 'STRING')
('fare', 'NUMERIC')
('cabin', 'STRING')
('embarked', ['C', 'Q', 'S'])
('boat', 'STRING')
('body', 'NUMERIC')
('home.dest', 'STRING')

Liczba obserwachji: 1309.


In [4]:
df.head(20)

,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1.0,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0.0,0.0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1.0,1,"Allison, Master. Hudson Trevor",male,0.9167,1.0,2.0,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1.0,0,"Allison, Miss. Helen Loraine",female,2.0000,1.0,2.0,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1.0,0,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1.0,2.0,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1.0,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1.0,2.0,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
5,1.0,1,"Anderson, Mr. Harry",male,48.0000,0.0,0.0,19952,26.5500,E12,S,3,NaN,"New York, NY"
6,1.0,1,"Andrews, Miss. Kornelia Theodosia",female,63.0000,1.0,0.0,13502,77.9583,D7,S,10,NaN,"Hudson, NY"
7,1.0,0,"Andrews, Mr. Thomas Jr",male,39.0000,0.0,0.0,112050,0.0000,A36,S,NaN,NaN,"Belfast, NI"
8,1.0,1,"Appleton, Mrs. Edward Dale (Charlotte Lamson)",female,53.0000,2.0,0.0,11769,51.4792,C101,S,D,NaN,"Bayside, Queens, NY"
9,1.0,0,"Artagaveytia, Mr. Ramon",male,71.0000,0.0,0.0,PC 17609,49.5042,NaN,C,NaN,22.0,"Montevideo, Uruguay"


**Liczba cech (features):**  
Zbiór zawiera 14 kolumn: **13 cech predykcyjnych** (`pclass`, `name`, `sex`, `age`, `sibsp`, `parch`, `ticket`, `fare`, `cabin`, `embarked`, `boat`, `body`, `home.dest`) oraz **1 zmienną celu** (`survived`). Zbiór liczy 1309 instancji.

**Typy danych cech:**

- Liczbowe ciągłe (`age`, `fare`) 
- Liczbowe dyskretne (`sibsp`, `parch`,`body`) 
- Kategoryczne (`sex`, `embarked`, `pclass`)
- Tekstowe o wysokiej kardynalności (`name`, `ticket`, `cabin`, `home.dest`,`boat`)

**Obserwacje:**
- Cechy `boat` oraz `body` silnie determinują przeżycie pasażera już na pierwszy rzut oka (rejestracja nastąpiła po katastrofie).
- Widoczne są liczne braki danych już w pierwszych 20 rekordach (`cabin`, `boat`, `body`).

### Analiza brakujących wartości

In [18]:
print(pd.isnull(df).sum()[pd.isnull(df).sum() > 0])
print()
print(pd.isnull(df).mean()[pd.isnull(df).sum() > 0])
cols_with_nan = list(pd.isnull(df).mean()[pd.isnull(df).sum() > 0].index)

age           263
fare            1
cabin        1014
embarked        2
boat          823
body         1188
home.dest     564
dtype: int64

age          0.200917
fare         0.000764
cabin        0.774637
embarked     0.001528
boat         0.628724
body         0.907563
home.dest    0.430863
dtype: float64


**Czy zbiór wymagał modyfikacji przed analizą braków?**

W badanym zbiorze braki danych występują w 7 z 14 kolumn. Ich obecność została wykryta za pomocą funkcji pd.isnull().sum() oraz pd.isnull().mean(). Podczas wczytywania pliku .arff parser arff.load() automatycznie zamienił źródłowe znaki zapytania (?) na None, dzięki czemu Pandas rozpoznał brakujące wartości i nadał zamianił je na np.nan.

W przypadku wczytywania danych bezpośrednio z surowego pliku tekstowego konieczna byłaby ręczna konwersja '?'( użytych w pliku źródłowym jako oznaczenia brakujących wartości) na np.nan. Można to zrobić za pomocą np. df.replace('?', np.nan) lub w pd.read_csv ustawiając odpowiednio parametr na_values='?'.

**Rozkład braków w cechach:**

- Duży udział braków (powyżej 60%): `body` (90,76%, 1188 braków), `cabin` (77,46%, 1014 braków) oraz `boat` (62,87%, 823 braki).

- Umiarkowany udział braków (20–45%): `home.dest` (43,09%, 564 braki) oraz `age` (20,09%, 263 braki). 

- Niewielki udział braków (poniżej 1%): `embarked` (0,15%, 2 braki) oraz `fare` (0,08%, 1 brak).

In [30]:
df.iloc[pd.isnull(df["embarked"]) | pd.isnull(df["fare"])]

,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
168,1.0,1,"Icard, Miss. Amelie",female,38.0,0.0,0.0,113572,80.0,B28,NaN,6,NaN,NaN
284,1.0,1,"Stone, Mrs. George Nelson (Martha Evelyn)",female,62.0,0.0,0.0,113572,80.0,B28,NaN,6,NaN,"Cincinatti, OH"
1225,3.0,0,"Storey, Mr. Thomas",male,60.5,0.0,0.0,3701,NaN,NaN,S,NaN,261.0,NaN


In [ ]:
df['body'].isnull().groupby(df['survived']).mean()

survived
0    0.850433
1    1.000000
Name: body, dtype: float64

Średnia zmiennej wskaźnikowej bodyNull dla ocalałych (*survived = 1*) wynosi dokładnie 100%, co oznacza, że żaden ocalały nie posiada identyfikatora ciała. Z kolei w grupie ofiar (*survived = 0*) wskaźnik braków wynosi 85%, co wskazuje, że ciał zdecydowanej większości zmarłych nigdy nie odnaleziono lub nie zidentyfikowano.

Brakujące dane w kolumnie body reprezentują typ brakujących danych *MNAR*, ponieważ stoi za tym oczywisty mechanizm przyczynowo skutkowy. 

Czy inne braki są zależne od kolumny survived?

In [ ]:
print(df[cols_with_nan].isnull().groupby(df['survived']).mean())

               age      fare     cabin  embarked      boat      body  \
survived                                                               
0         0.234858  0.001236  0.873918     0.000  0.988875  0.850433   
1         0.146000  0.000000  0.614000     0.004  0.046000  1.000000   

          home.dest  
survived             
0          0.508035  
1          0.306000  


Kolejną kolumną MNAR okazuje się kolumna boat, ofiary w większości nie dostały się na żadną z szalup ratunkowych, co jak było spowodowane niedostateczną liczbą szalup na pokładzie.

In [21]:
print(df[cols_with_nan].isnull().groupby(df['pclass']).mean())

             age     fare     cabin  embarked      boat      body  home.dest
pclass                                                                      
1.0     0.120743  0.00000  0.207430  0.006192  0.377709  0.891641   0.105263
2.0     0.057762  0.00000  0.916968  0.000000  0.595668  0.888087   0.057762
3.0     0.293371  0.00141  0.977433  0.000000  0.755994  0.922426   0.724965


In [46]:
pd.concat([df["survived"].astype(int),df["pclass"]],axis=1).groupby('pclass').mean()

,survived
pclass,
1.0,0.619195
2.0,0.429603
3.0,0.255289


Nieprzypadkowe są również braki dla kolumn home.dest oraz cabin. Wynika to ze względu na to, że w ocalałych listach pasażerów numer kabin w ogóle nie był wpisywany, a pełne plany zakwaterowania zatonęły wraz ze statkiem. W efekcie dane te pochodzą niemal wyłącznie ze wspomnień ocalałych (oraz zachowanej częściowej listy stewarda dla 1. klasy), przy czym część osób mogła celowo nie ujawniać numeru bądź go nie pamiętać. Dodatkowo dla 3. klasy nie było sztywnego przydziału kabin przed wypłynięciem i były one przydzielane dopiero na pokładzie. 

Braki docelowego kierunku podróży wynikają natomiast z tego, że na oficjalnych listach dla pasażerów z przesiadkami nie rejestrowano ich adresu docelowego w Ameryce, a jedynie port wyokrętowania samego Titanica. Jednak dla ocalałych,co również ma bezpośredni związek z klasą pasażera, udawało się ustalić dokąd zmierzali, a nawet dla ofiar z 1. lub 2. klasy, które były z reguły dobrze usytuowane i znane, badacze byli w stanie w większości dociec celu ich podróży np. na podstawie przekazów bliskich.

### Rodzaje występujących typów braków danych
1. **MNAR (Missing Not at Random):**
   * **`body`:** 100% braków u ocalałych i 85.0% u ofiar. Brak wynika z faktu braku zgonu lub niewyłowienia ciała z oceanu.
   * **`boat`:** 98.9% braków u ofiar vs 4.6% u ocalałych. Brak numeru szalupy bezpośrednio oznacza, że pasażer nie znalazł się w łodzi ratunkowej.
2. **MAR (Missing at Random):**
   * **`cabin`:** Odsetek braków silnie skorelowany z klasą biletu (`pclass` = 3 to ponad 95% braków kabin).
   * **`home.dest`:** 72.5% braków w 3. klasie wobec zaledwie 5-10% w 1. i 2. klasie. Mechanizm wynika z procedur rejestracji emigrantów.
   * **`age`:** W 3. klasie częściej brakowało dokumentów tożsamości niż w klasach wyższych.
3. **MCAR (Missing Completely at Random):**
   * **`fare`** (1 brak) oraz **`embarked`** (2 braki): Losowe błędy pisarskie/rejestracyjne, niezależne od losu pasażera ani innych zmiennych.

### Postępowanie z brakującymi wartościami (Zadanie 4)
* **`boat`, `body` -> Usunięcie kolumn (ochrona przed Data Leakage):**
  Informacje te powstały w trakcie lub po katastrofie. Użycie ich w modelu predykcyjnym sztucznie zawyżyłoby skuteczność predykcji.
* **`cabin` -> Inżynieria cech (Feature Engineering):**
  Ze względu na 77.5% braków imputacja konkretnego numeru nie ma sensu. W feature enginerringu dobrą decyzją by było utworzenie flagi binarnej `has_cabin` (0/1).
* **`age` -> Imputacja warunkowa:**
  Kluczowa zmienna biologiczna (20% braków). Usunięcie wierszy spowodowałoby utratę cennych danych. Rekomendowana imputacja medianą w podgrupach (wyznaczonych przez `pclass` i tytuł wyciągnięty z nazwiska, np. *Mr*, *Miss*, *Master*) lub algorytmem KNNImputer.
* **`embarked`, `fare` -> Prosta imputacja:**
  * `embarked`: Imputacja modą (najczęstszy port: `'S'`).
  * `fare`: Imputacja medianą dla pasażerów 3. klasy.
* **`home.dest` -> Usunięcie cechy:**
  Wysoka kardynalność i ponad 43% braków – cecha o małej przydatności w standardowej klasyfikacji tabelarycznej.